# 04 — Perfis dos Clusters e Ações

Objetivos:
- Anexar `cluster_id` à visão cliente-atual
- Construir perfis (tamanho, percentuais, médias/medianas, distribuições)
- Nomear clusters com rótulos de negócio
- Criar ranking (valor/risco) com métrica transparente
- Responder explicitamente: recorrente e nunca acessou
- Exportar: `dataset_clusterizado.xlsx` e `resumo_clusters.xlsx`

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))


import numpy as np
import pandas as pd

from src.features import build_modeling_dataframe
from src.modeling import (
    cluster_profiles,
    compute_cluster_score,
    load_pipeline,
)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_cliente = pd.read_parquet(PROCESSED_DIR / 'base_cliente_atual.parquet')
X, spec = build_modeling_dataframe(df_cliente)
pipe = load_pipeline(str(REPORTS_DIR / 'kmeans_pipeline.joblib'))
df_cliente.shape, X.shape

## 1) Predição de cluster

In [ ]:
df_cliente = df_cliente.copy()
df_cliente['cluster_id'] = pipe.predict(X)
df_cliente['cluster_id'].value_counts().sort_index()

## 2) Perfis e nomes de negócio

In [ ]:
NOME_BY_CLUSTER = {
    4: 'Champions',
    0: 'Potenciais',
    5: 'Novos',
    1: 'Avulsos Engajados',
    3: 'Churn Iminente',
    2: 'Zumbis',
}

df_cliente['cluster_id'] = df_cliente['cluster_id'].astype(int)
names = (
    pd.DataFrame({'cluster_id': list(NOME_BY_CLUSTER.keys()), 'nome_cluster': list(NOME_BY_CLUSTER.values())})
    .sort_values('cluster_id')
    .reset_index(drop=True)
)

df_cliente['nome_cluster'] = df_cliente['cluster_id'].map(NOME_BY_CLUSTER)
if df_cliente['nome_cluster'].isna().any():
    missing = sorted(df_cliente.loc[df_cliente['nome_cluster'].isna(), 'cluster_id'].unique().tolist())
    raise ValueError(f'Clusters sem nome mapeado: {missing}')

names

In [ ]:
profiles = cluster_profiles(
    df_cliente,
    cluster_col='cluster_id',
    numeric_cols=[c for c in ['log_acessos','dias_sem_acessar','recorrente','total_parcelas','n_transacoes_cliente','recencia_compra_dias','freq_compra_mensal','parcelado','nunca_acessou'] if c in df_cliente.columns],
    categorical_cols=[c for c in ['ativo','renovacao','pagamento_tratado','tipo_pagamento','faixa_inatividade','status'] if c in df_cliente.columns],
)
profiles['summary'].head(10)

## 3) Ranking (valor/risco)

Métrica transparente: score ponderado (engajamento + recorrência + renovação − inatividade − nunca_acessou).

In [ ]:
rank = compute_cluster_score(df_cliente, cluster_col='cluster_id')
rank = rank.merge(names, on='cluster_id', how='left')
rank

## 4) Pergunta 4: recorrente e nunca acessou

Filtro na visão cliente-atual: `recorrente==1` e `nunca_acessou==1`.

In [ ]:
mask = (df_cliente.get('recorrente', 0).fillna(0) == 1) & (df_cliente.get('nunca_acessou', 0).fillna(0) == 1)
q4 = df_cliente.loc[mask].copy()
q4_size = len(q4)
q4_pct = q4_size / len(df_cliente)
q4_size, q4_pct

## 5) Ações recomendadas por cluster (playbooks)

As recomendações abaixo são geradas a partir do perfil (engajamento/inatividade/recorrência/parcelamento). Ajuste fino pode ser feito após leitura do dicionário do case (PDF).

In [ ]:
def recomendar_acao(row):
    nome = str(row['nome_cluster'])
    if nome == 'Champions':
        return 'Upsell/cross-sell, programa de indicação, early access; manter NPS alto com comunicações personalizadas.'
    if nome == 'Potenciais':
        return 'Elevar valor percebido e reduzir inatividade: campanhas por interesse/curso e check-ins por gatilho (14–30 dias sem acessar).'
    if nome == 'Novos':
        return 'Boas-vindas, checklist de primeiros passos, gatilhos de valor (primeira aula/primeiro resultado), acompanhamento D+3/D+7.'
    if nome == 'Avulsos Engajados':
        return 'Converter para recorrência: oferta de plano recorrente com proposta clara, bundles e condições especiais para upgrade.'
    if nome == 'Churn Iminente':
        return 'Playbook de retenção: diagnóstico de bloqueios, campanhas win-back, ofertas de pausa/alternativa de plano, CS 1:1 para top contas.'
    if nome == 'Zumbis':
        return 'Onboarding agressivo + nudges de ativação (sequência 7 dias), webinars, contato CS proativo; foco em ativação para reduzir cancelamento futuro.'
    return 'Ações de engajamento: trilhas de conteúdo, lembretes de progresso, campanhas por faixa de inatividade e tipo de pagamento.'

acoes = names.copy()
acoes['acoes_recomendadas'] = acoes.apply(recomendar_acao, axis=1)
acoes

## 6) Exportação obrigatória (Excel)
- `reports/dataset_clusterizado.xlsx`: base cliente-atual + cluster + nome_cluster + principais features
- `reports/resumo_clusters.xlsx`: tabelas agregadas + ranking + ações

In [ ]:
main_cols = [c for c in ['cliente','cluster_id','nome_cluster','log_acessos','dias_sem_acessar','recorrente','total_parcelas','parcelado','nunca_acessou','renovacao','ativo','n_transacoes_cliente','recencia_compra_dias','freq_compra_mensal','tipo_pagamento','pagamento_tratado','status'] if c in df_cliente.columns]
dataset_clusterizado = df_cliente[main_cols].copy()

dataset_path = REPORTS_DIR / 'dataset_clusterizado.xlsx'
with pd.ExcelWriter(dataset_path, engine='xlsxwriter') as writer:
    dataset_clusterizado.to_excel(writer, sheet_name='cliente_atual_cluster', index=False)
    q4[main_cols].to_excel(writer, sheet_name='q4_recorrente_sem_acesso', index=False)

resumo_path = REPORTS_DIR / 'resumo_clusters.xlsx'
with pd.ExcelWriter(resumo_path, engine='xlsxwriter') as writer:
    profiles['summary'].to_excel(writer, sheet_name='perfil_resumo', index=False)
    profiles['sizes'].to_excel(writer, sheet_name='tamanhos', index=False)
    profiles['numeric'].to_excel(writer, sheet_name='numericas', index=False)
    profiles['categorical'].to_excel(writer, sheet_name='categoricas', index=False)
    rank.to_excel(writer, sheet_name='ranking', index=False)
    acoes.to_excel(writer, sheet_name='acoes', index=False)

df_cliente.to_parquet(PROCESSED_DIR / 'base_cliente_clusterizada.parquet', index=False)
dataset_path, resumo_path